In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import entropy

# ----------------------------
# Utility functions
# ----------------------------
def confidence_weighted_f1(y_true, y_pred, confidence):
    weighted_tp = np.sum(confidence * (y_pred == 1) * (y_true == 1))
    weighted_fp = np.sum(confidence * (y_pred == 1) * (y_true == 0))
    weighted_fn = np.sum(confidence * (y_pred == 0) * (y_true == 1))
    precision = weighted_tp / (weighted_tp + weighted_fp) if (weighted_tp + weighted_fp) > 0 else 0
    recall = weighted_tp / (weighted_tp + weighted_fn) if (weighted_tp + weighted_fn) > 0 else 0
    return 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

def multiclass_confidence_weighted_f1(y_true, y_pred, y_proba, average='macro'):
    classes = np.unique(y_true)
    f1_scores = []
    supports = []

    for cls in classes:
        y_true_bin = (y_true == cls).astype(int)
        y_pred_bin = (y_pred == cls).astype(int)
        # Ensure y_proba has shape (n_samples, n_classes)
        if y_proba.ndim == 1:
            y_proba = y_proba.reshape(-1, 1)
        if y_proba.shape[1] <= cls:
            # Add dummy column if some class missing
            y_proba = np.hstack([y_proba, np.zeros((y_proba.shape[0], cls - y_proba.shape[1] + 1))])
        conf = np.where(y_pred == cls, np.max(y_proba, axis=1), 0)
        f1_c = confidence_weighted_f1(y_true_bin, y_pred_bin, conf)
        f1_scores.append(f1_c)
        supports.append(np.sum(y_true == cls))

    f1_scores = np.nan_to_num(f1_scores)
    supports = np.array(supports)
    
    if average == 'macro':
        return np.mean(f1_scores)
    elif average == 'weighted':
        return np.sum(f1_scores * supports) / np.sum(supports)
    else:
        raise ValueError("average must be 'macro' or 'weighted'")



class IndustryClassifier:
    def __init__(self, random_state=42):
        # Fixed hyperparameters
        self.n_neighbors = 10
        self.n_components = 75
        self.alpha = 0.65
        self.random_state = random_state

        # Placeholders
        self.label_encoder = LabelEncoder()
        self.pca = PCA(n_components=self.n_components, random_state=self.random_state)
        self.knn_cal = None
        self.hgb_cal = None
        self.fin_preproc = None
        self.fin_numeric_cols = ['net_profit_margin', 'asset_turnover', 'netprofit_asset', 'netprofit_over_asset']
        self.fin_cat_cols = ['country_code']

    def _expand_embeddings(self, series):
        return np.vstack(series.apply(lambda x: np.array(list(map(float, ast.literal_eval(x))))))

    def fit(self, path_text="data/df_train.pkl", path_fin="data/df_financials_train.pkl"):
        # Load and merge
        text_embeddings = pd.read_pickle(path_text)
        fin_data = pd.read_pickle(path_fin)
        text_embeddings['industry'] = self.label_encoder.fit_transform(text_embeddings['industry'])
        df = pd.merge(text_embeddings, fin_data, on='id', how='left')

        # Clean numeric financials
        financial_numeric_cols = ['net_profit_margin', 'asset_turnover']
        for col in financial_numeric_cols:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = np.log1p(df[col].clip(lower=0))

        # Interactions
        df['netprofit_asset'] = df['net_profit_margin'] * df['asset_turnover']
        df['netprofit_over_asset'] = df['net_profit_margin'] / (df['asset_turnover'] + 1e-6)
        self.fin_numeric_cols = financial_numeric_cols + ['netprofit_asset', 'netprofit_over_asset']

        # Embeddings
        self.emb_train = self._expand_embeddings(df['business_description_embedding'])
        self.emb_train_pca = self.pca.fit_transform(self.emb_train)

        # Financial preprocessing
        numeric_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        categorical_transformer = Pipeline([
            ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ])
        self.fin_preproc = ColumnTransformer([
            ('num', numeric_transformer, self.fin_numeric_cols),
            ('cat', categorical_transformer, self.fin_cat_cols)
        ])
        X_train_fin = self.fin_preproc.fit_transform(df[self.fin_numeric_cols + self.fin_cat_cols])
        ros = RandomOverSampler(random_state=self.random_state)
        X_train_fin_res, y_train_fin_res = ros.fit_resample(X_train_fin, df['industry'].values)

        # KNN + calibrated probabilities
        knn = KNeighborsClassifier(n_neighbors=self.n_neighbors, weights='distance', n_jobs=-1)
        self.knn_cal = CalibratedClassifierCV(knn, method='sigmoid', cv=3, n_jobs=-1)
        self.knn_cal.fit(self.emb_train_pca, df['industry'].values)

        # Financial HGB + calibrated probabilities
        hgb = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, class_weight='balanced', random_state=self.random_state)
        self.hgb_cal = CalibratedClassifierCV(hgb, method='isotonic', cv=3, n_jobs=-1)
        self.hgb_cal.fit(X_train_fin_res, y_train_fin_res)

        self.n_classes = max(self.knn_cal.classes_.size, self.hgb_cal.classes_.size)

    def predict(self, df_input, labels_decoded=False):
        # Embeddings
        emb_test = self._expand_embeddings(df_input['business_description_embedding'])
        emb_test_pca = self.pca.transform(emb_test)

        # Text model probabilities
        y_proba_text = self.knn_cal.predict_proba(emb_test_pca)

        # Financial model probabilities
        X_fin = self.fin_preproc.transform(df_input[self.fin_numeric_cols + self.fin_cat_cols])
        y_proba_fin = self.hgb_cal.predict_proba(X_fin)

        # Align class dimensions
        if y_proba_text.shape[1] < self.n_classes:
            y_proba_text = np.hstack([y_proba_text, np.zeros((y_proba_text.shape[0], self.n_classes - y_proba_text.shape[1]))])
        if y_proba_fin.shape[1] < self.n_classes:
            y_proba_fin = np.hstack([y_proba_fin, np.zeros((y_proba_fin.shape[0], self.n_classes - y_proba_fin.shape[1]))])

        # Alpha fusion
        y_proba_final = self.alpha * y_proba_text + (1 - self.alpha) * y_proba_fin
        y_pred_encoded = np.argmax(y_proba_final, axis=1)

        # Hybrid confidence
        agreement = (np.argmax(y_proba_text, axis=1) == np.argmax(y_proba_fin, axis=1)).astype(float)
        base_confidence = np.max(y_proba_final, axis=1)
        ent = entropy(y_proba_final.T)
        ent_norm = (ent - ent.min()) / (ent.max() - ent.min() + 1e-9)
        entropy_penalty = 1 - ent_norm
        y_confidence = base_confidence * entropy_penalty * (0.5 + 0.5 * agreement)

        if labels_decoded:
            y_pred_decoded = self.label_encoder.inverse_transform(y_pred_encoded)
            return y_pred_decoded, y_confidence
        else:
            return y_pred_encoded, y_confidence

### Example usage: ###
# Initialize and train
clf = IndustryClassifier()
clf.fit()

# Predict encoded labels and confidence
y_pred_encoded, y_conf = clf.predict(df_test)
print("Encoded labels:", y_pred_encoded[:10])
print("Confidence:", y_conf[:10])

# Predict decoded industry names, encoded labels, and confidence
y_pred_decoded, y_conf = clf.predict(df_test, labels_decoded=True)
print("Decoded industries:", y_pred_decoded[:10])
print("Encoded labels:", y_pred_encoded[:10])
print("Confidence:", y_conf[:10])


In [ ]:
# ----------------------------
# Load data
# ----------------------------
text_embeddings = pd.read_pickle("data/df_train.pkl")
fin_data = pd.read_pickle("data/df_financials_train.pkl")

# Label encode target
le = LabelEncoder()
text_embeddings['industry'] = le.fit_transform(text_embeddings['industry'])

# Merge datasets
df = pd.merge(text_embeddings, fin_data, on='id', how='left')

# Financial columns
financial_numeric_cols = ['net_profit_margin', 'asset_turnover']
financial_cat_cols = ['country_code']

# Clean numeric financials
for col in financial_numeric_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = np.log1p(df[col].clip(lower=0))

# Add interactions
df['netprofit_asset'] = df['net_profit_margin'] * df['asset_turnover']
df['netprofit_over_asset'] = df['net_profit_margin'] / (df['asset_turnover'] + 1e-6)
financial_numeric_cols += ['netprofit_asset', 'netprofit_over_asset']

# ----------------------------
# Train / Test split
# ----------------------------
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['industry'], random_state=42)
y_test = df_test['industry'].values

# ----------------------------
# Expand embeddings
# ----------------------------
def expand_embeddings(series):
    return np.vstack(series.apply(lambda x: np.array(list(map(float, ast.literal_eval(x))))))

emb_train = expand_embeddings(df_train['business_description_embedding'])
emb_test = expand_embeddings(df_test['business_description_embedding'])

# PCA
pca = PCA(n_components=75, random_state=42)
emb_train_pca = pca.fit_transform(emb_train)
emb_test_pca = pca.transform(emb_test)

# ----------------------------
# Oversample training set (financial branch)
# ----------------------------
ros = RandomOverSampler(random_state=42)

# Financial preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])
fin_preproc = ColumnTransformer([
    ('num', numeric_transformer, financial_numeric_cols),
    ('cat', categorical_transformer, financial_cat_cols)
])

X_train_fin = fin_preproc.fit_transform(df_train[financial_numeric_cols + financial_cat_cols])
X_test_fin = fin_preproc.transform(df_test[financial_numeric_cols + financial_cat_cols])
X_train_fin_res, y_train_fin_res = ros.fit_resample(X_train_fin, df_train['industry'].values)

# ----------------------------
# Text model: KNN + calibrated probabilities
# ----------------------------
knn = KNeighborsClassifier(n_neighbors=10, weights='distance', n_jobs=-1)
knn_cal = CalibratedClassifierCV(knn, method='sigmoid', cv=3, n_jobs=-1)
knn_cal.fit(emb_train_pca, df_train['industry'].values)
y_proba_text = knn_cal.predict_proba(emb_test_pca)
y_pred_text = np.argmax(y_proba_text, axis=1)

# ----------------------------
# Financial model: HGB + calibrated probabilities
# ----------------------------
hgb = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, class_weight='balanced', random_state=42)
hgb_cal = CalibratedClassifierCV(hgb, method='isotonic', cv=3, n_jobs=-1)
hgb_cal.fit(X_train_fin_res, y_train_fin_res)
y_proba_fin = hgb_cal.predict_proba(X_test_fin)
y_pred_fin = np.argmax(y_proba_fin, axis=1)

# ----------------------------
# Alpha-weighted fusion
# ----------------------------
alpha = 0.65
# Ensure same number of classes
n_classes = max(y_proba_text.shape[1], y_proba_fin.shape[1])
if y_proba_text.shape[1] < n_classes:
    y_proba_text = np.hstack([y_proba_text, np.zeros((y_proba_text.shape[0], n_classes - y_proba_text.shape[1]))])
if y_proba_fin.shape[1] < n_classes:
    y_proba_fin = np.hstack([y_proba_fin, np.zeros((y_proba_fin.shape[0], n_classes - y_proba_fin.shape[1]))])

y_proba_final = alpha * y_proba_text + (1 - alpha) * y_proba_fin
y_pred_final = np.argmax(y_proba_final, axis=1)

# ----------------------------
# Hybrid confidence: calibrated * agreement * entropy penalty
# ----------------------------
agreement = (y_pred_text == y_pred_fin).astype(float)
base_confidence = np.max(y_proba_final, axis=1)
ent = entropy(y_proba_final.T)
ent_norm = (ent - ent.min()) / (ent.max() - ent.min() + 1e-9)
entropy_penalty = 1 - ent_norm
confidence_final = base_confidence * entropy_penalty * (0.5 + 0.5 * agreement)

# ----------------------------
# Evaluate
# ----------------------------
cw_f1_macro = multiclass_confidence_weighted_f1(y_test, y_pred_final, confidence_final, average='macro')
cw_f1_weighted = multiclass_confidence_weighted_f1(y_test, y_pred_final, confidence_final, average='weighted')

print(f"Final results on TEST set:")
print("Confidence-weighted F1 (macro):", round(cw_f1_macro, 4))
print("Confidence-weighted F1 (weighted):", round(cw_f1_weighted, 4))
